In [ ]:
import subprocess
subprocess.run(['pip', 'install', 'scanpy', 'cell2location', '-q'], check=True)
print('Dependencies installed.')

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import scipy.io
import cell2location
from cell2location.models import RegressionModel, Cell2location
from scipy.sparse import issparse, csr_matrix
import gc, os

BASE = '/kaggle/input/spatial-transcriptomics-dataset'
REF_DIR = BASE + '/data/reference/Wu_etal_2021_BRCA_scRNASeq'
OUT_DIR = '/kaggle/working'

print('Step 1: Downloading raw spatial data...')
os.system('wget -q "https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_1/V1_Breast_Cancer_Block_A_Section_1_filtered_feature_bc_matrix.h5" -O /kaggle/working/raw.h5')

adata = sc.read_10x_h5('/kaggle/working/raw.h5')
adata.var_names_make_unique()
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
X = adata.X.copy()
if issparse(X):
    X.data = np.round(X.data).astype(np.int32)
adata.layers['counts'] = X
print(f'Spots: {adata.n_obs}, Genes: {adata.n_vars}')

In [ ]:
print('Step 2: Loading reference...')
meta = pd.read_csv(REF_DIR + '/metadata.csv', index_col=0)
barcodes = pd.read_csv(REF_DIR + '/count_matrix_barcodes.tsv', header=None)
np.random.seed(42)
selected = []
for ct in meta['celltype_major'].unique():
    idx = meta[meta['celltype_major'] == ct].index.tolist()
    selected.extend(np.random.choice(idx, min(2000, len(idx)), replace=False))
sel_set = set(selected)
sel_idx = [i for i, b in enumerate(barcodes[0]) if b in sel_set]
mat = scipy.io.mmread(REF_DIR + '/count_matrix_sparse.mtx').T.tocsr()
mat_sub = mat[sel_idx, :].copy()
del mat
gc.collect()
genes = pd.read_csv(REF_DIR + '/count_matrix_genes.tsv', header=None)
ref = sc.AnnData(X=csr_matrix(mat_sub))
ref.obs_names = barcodes[0].values[sel_idx]
ref.var_names = genes[0].values
ref.obs['celltype_major'] = meta.loc[ref.obs_names, 'celltype_major'].values
X_ref = ref.X.copy()
X_ref.data = np.round(X_ref.data).astype(np.int32)
ref.layers['counts'] = X_ref
print(ref.obs['celltype_major'].value_counts())

In [ ]:
print('Step 3: Common genes...')
sc.pp.filter_genes(ref, min_cells=5)
common = list(adata.var_names.intersection(ref.var_names))
adata = adata[:, common].copy()
ref   = ref[:, common].copy()
print(f'Common genes: {len(common)}')

print('Step 4: Training reference model...')
RegressionModel.setup_anndata(ref, labels_key='celltype_major', layer='counts')
reg_model = RegressionModel(ref)
reg_model.train(max_epochs=200, batch_size=2500, accelerator='gpu')

In [ ]:
print('Step 5: Signatures...')
ref = reg_model.export_posterior(ref, sample_kwargs={'num_samples':500, 'batch_size':2500, 'accelerator':'gpu'})
sig = np.asarray(ref.varm['means_per_cluster_mu_fg'])
factor_names = list(ref.uns['mod']['factor_names'])
inf_aver = pd.DataFrame(sig if sig.shape[0] == ref.n_vars else sig.T, index=ref.var_names, columns=factor_names).astype(np.float32)
del ref, reg_model
gc.collect()
print(f'Signatures: {inf_aver.shape}')

print('Step 6: Spatial model...')
Cell2location.setup_anndata(adata, layer='counts')
c2l = Cell2location(adata, cell_state_df=inf_aver, N_cells_per_location=8, detection_alpha=20)
c2l.train(max_epochs=300, batch_size=None, accelerator='gpu')

In [ ]:
print('Step 7: Saving...')
adata = c2l.export_posterior(adata, sample_kwargs={'num_samples':500, 'batch_size':adata.n_obs, 'accelerator':'gpu'})
prop_df = adata.obsm['q05_cell_abundance_w_sf'].copy()
prop_df.columns = [c.replace('q05cell_abundance_w_sf_', '') for c in prop_df.columns]
prop_df.to_csv(OUT_DIR + '/cell_proportions.csv')
print('\nAverage cell abundances:')
print(prop_df.mean().sort_values(ascending=False).round(3))
print('\nDeconvolution Complete!')